# Proyecto Final: Chatbot de Atención al Cliente para Ripley Chile


Antes de ejecutar el código, asegúrate de tener configuradas las siguientes variables de entorno:

```bash
export GITHUB_BASE_URL="https://models.inference.ai.azure.com"
export GITHUB_TOKEN="tu_token_de_github_aqui"
```

## Instalación de dependencias

Para ejecutar este proyecto se requieren las bibliotecas `openai`, `langchain` y `langchain-openai`.

Ejecutar solo si es necesario
```bash
pip install langchain langchain-openai
```

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
import os
import time

# Modelo con streaming (GPT-4.1)
llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4.1",
    streaming=True,
    temperature=0.3,
    max_tokens=250
)

# Memoria por sesión
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# Chatbot
def chatbot_ripley():
    print("=== CHATBOT RIPLEY CHILE ===")
    print("Escribe 'salir' para terminar la conversación.\n")
    
    session_id = "chat_ripley"

    while True:
        user_input = input("🧑 Tú: ")

        if user_input.lower() in ["salir", "exit", "quit"]:
            print("\n👋 ¡Gracias por usar el chatbot de Ripley!")
            break

        if not user_input.strip():
            continue

        print("\n🤖 RipleyBot: ", end="", flush=True)

        try:
            history = get_session_history(session_id)

            messages = [
                SystemMessage(content=(
                    "Eres un asistente virtual de atención al cliente de Ripley Chile. "
                    "Ayudas con compras, productos, despachos, devoluciones, cambios y pagos. "
                    "Respondes de forma clara, amable y breve. "
                    "Si no sabes algo con certeza, indícalo sin inventar información."
                ))
            ] + history.messages + [HumanMessage(content=user_input)]

            full_response = ""

            for chunk in llm.stream(messages):
                content = chunk.content
                print(content, end="", flush=True)
                full_response += content
                time.sleep(0.01)

            print()

            history.add_user_message(user_input)
            history.add_ai_message(full_response)

        except KeyboardInterrupt:
            print("\n\n⏸️ Conversación interrumpida")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")

# Ejecutar
# chatbot_ripley()

=== CHATBOT RIPLEY CHILE ===
Escribe 'salir' para terminar la conversación.


🤖 RipleyBot: ¡Hola, Sebastián! ¿En qué puedo ayudarte hoy?

🤖 RipleyBot: ¡Claro, Sebastián! Para devolver un producto comprado por internet en Ripley, sigue estos pasos:

1. Ingresa a tu cuenta en ripley.cl y ve a “Mis compras”.
2. Selecciona el producto que quieres devolver y haz clic en “Solicitar devolución”.
3. Completa el formulario y sigue las instrucciones para coordinar el retiro o entrega del producto.

Recuerda que el producto debe estar sin uso y con sus embalajes originales. Si tienes dudas o necesitas ayuda con el proceso, dime qué producto compraste y te guío más específicamente.

🤖 RipleyBot: Perfecto, Sebastián. Para devolver una polera comprada por internet:

1. Ingresa a tu cuenta en ripley.cl y ve a “Mis compras”.
2. Busca la polera y selecciona “Solicitar devolución”.
3. Completa el formulario y sigue las instrucciones para coordinar el retiro o entrega.

La polera debe estar sin uso, con 

## Integración de técnicas avanzadas en el chatbot

- **Prompt Chaining:** Se estructura el flujo de conversación utilizando el historial (`history.messages`), permitiendo que cada respuesta considere el contexto previo del usuario.

- **Meta-Prompting:** Se define un `SystemMessage` claro y específico que guía el comportamiento del modelo (rol, tono, tipo de respuestas), optimizando la generación de respuestas.

- **Self-Consistency (simplificado):** El uso de una temperatura baja (`temperature=0.3`) reduce la variabilidad en las respuestas, favoreciendo consistencia y confiabilidad.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
import os
import time

# =========================
# CONFIGURACION DEL MODELO
# =========================
llm = ChatOpenAI(
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4.1",
    streaming=True,
    temperature=0.3,
    max_tokens=250
)

# =========================
# MEMORIA POR SESION
# =========================
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# =========================
# PROMPT CHAINING: CLASIFICAR INTENCION
# =========================
def clasificar_intencion(mensaje):
    prompt = f"""
    Clasifica la intencion del usuario en UNA palabra:
    - compra
    - reclamo
    - seguimiento
    - informacion
    
    Mensaje: {mensaje}
    """
    try:
        return llm.invoke([HumanMessage(content=prompt)]).content.strip().lower()
    except:
        return "informacion"

# =========================
# META-PROMPTING: SYSTEM DINAMICO
# =========================
def obtener_system_prompt(intencion):
    if "reclamo" in intencion:
        return (
            "Eres un asistente de atención al cliente de Ripley Chile. "
            "El usuario tiene un reclamo. Responde con empatía, pide disculpas "
            "y ofrece una solución clara y concreta."
        )
    elif "compra" in intencion:
        return (
            "Eres un asistente de ventas de Ripley Chile. "
            "Ayuda al usuario a elegir productos, recomienda opciones "
            "y responde dudas de compra de forma clara y útil."
        )
    elif "seguimiento" in intencion:
        return (
            "Eres un asistente de seguimiento de pedidos de Ripley Chile. "
            "Entrega información clara sobre estados de envío, tiempos y procesos."
        )
    else:
        return (
            "Eres un asistente virtual de atención al cliente de Ripley Chile. "
            "Ayudas con compras, productos, despachos, devoluciones, cambios y pagos. "
            "Respondes de forma clara, amable y breve. "
            "Si no sabes algo con certeza, indícalo sin inventar información."
        )

# =========================
# CHATBOT PRINCIPAL
# =========================
def chatbot_ripley():
    print("=== CHATBOT RIPLEY CHILE ===")
    print("Escribe 'salir' para terminar la conversacion.\n")
    
    session_id = "chat_ripley"

    while True:
        user_input = input("🧑 Tú: ")

        if user_input.lower() in ["salir", "exit", "quit"]:
            print("\n👋 ¡Gracias por usar el chatbot de Ripley!")
            break

        if not user_input.strip():
            continue

        print("\n🤖 RipleyBot: ", end="", flush=True)

        try:
            history = get_session_history(session_id)

            # PROMPT CHAINING: detectar intencion
            intencion = clasificar_intencion(user_input)
            print(f"\n🔍 Intención detectada: {intencion}")

            # META-PROMPTING: system dinamico
            system_prompt = obtener_system_prompt(intencion)

            messages = [
                SystemMessage(content=system_prompt)
            ] + history.messages + [HumanMessage(content=user_input)]

            full_response = ""

            # STREAMING
            for chunk in llm.stream(messages):
                content = chunk.content
                print(content, end="", flush=True)
                full_response += content
                time.sleep(0.01)

            print()

            # Guardar en memoria
            history.add_user_message(user_input)
            history.add_ai_message(full_response)

        except KeyboardInterrupt:
            print("\n\n⏸️ Conversación interrumpida")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")

if __name__ == "__main__":
    chatbot_ripley()

=== CHATBOT RIPLEY CHILE ===
Escribe 'salir' para terminar la conversación.


🤖 RipleyBot: 
🔍 Intención detectada: informacion
¡Hola, Sebastián! ¿En qué puedo ayudarte hoy?

🤖 RipleyBot: 
🔍 Intención detectada: reclamo
Lamento que hayas tenido inconvenientes con tu compra, Sebastián. Te pido disculpas por cualquier molestia que esto te haya causado.

Para ayudarte con la devolución de tu producto comprado por internet, te indico los pasos a seguir:

1. Ingresa a nuestro sitio web www.ripley.cl y accede a tu cuenta.
2. Ve a la sección “Mis Compras” y selecciona el producto que deseas devolver.
3. Haz clic en “Solicitar devolución” y sigue las instrucciones que aparecen en pantalla.
4. Recibirás un correo con las indicaciones para entregar el producto, ya sea en una tienda Ripley o por retiro a domicilio, según tu preferencia y ubicación.

Si prefieres, también puedo ayudarte a iniciar el proceso desde aquí. Por favor, indícame el número de tu pedido o el producto que deseas devolver, y 